# 🚀 DOENet: High-Dimensional Benchmark & Functional ANOVA
**Author:** Ho-joong Kim (DSLab) & Pickle  
**Repository:** `hojoongkimm/ai-ideas-lab`  
**Topic:** Differentiable Orthogonal Factor Screening & Functional ANOVA for End-to-End High-Dimensional Deep Learning

---
### 📌 Research Objectives
1. **High-Dimensional Multicollinear Robustness:** Verify if DOENet can prune noise/confounding features and retain the true causal features.
2. **Comparative Benchmark:** Compare DOENet against Standard MLP, L1-Sparse (LassoNet-style), and Gated MLP (w/o Ortho/ANOVA).
3. **Explainable ANOVA Decomposition:** Decompose predictions into Main Effects, Heredity-Gated Pairwise Interactions, and Deep Residuals.

In [ ]:
import os
import sys
import time
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

from models.doe_net import DOENet
from models.baselines import StandardMLP, L1SparseMLP, GatedMLP_Ablation
from datasets.synthetic_highdim import get_dataloaders
from utils.metrics import compute_regression_metrics, compute_feature_selection_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. High-Dimensional Synthetic Benchmark Dataset (P=500, N=2500)

In [ ]:
N_FEATURES = 500
N_SAMPLES = 2500

data = get_dataloaders(n_samples=N_SAMPLES, n_features=N_FEATURES, batch_size=64, seed=42)
train_loader = data['train']
val_loader = data['val']
test_loader = data['test']
true_indices = data['true_indices']
X_test = data['X_test'].to(device)
y_test = data['y_test'].to(device)

print(f'Total features: {N_FEATURES}')
print(f'Ground-truth active causal features: {true_indices} (10 of {N_FEATURES})')
print(f'Collinear noise features: 10..49')
print(f'Pure Gaussian noise features: 50..{N_FEATURES-1}')

## 2. Train DOENet (End-to-End Task + Orthogonality + Sparsity)

In [ ]:
model = DOENet(in_features=N_FEATURES, num_factors=24, hidden_dim=64, interaction_rank=16).to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80, eta_min=1e-5)

epochs = 80
history = {'loss': [], 'task_loss': [], 'sparse_loss': [], 'ortho_loss': []}

for epoch in range(1, epochs + 1):
    model.train()
    ep_loss, ep_task, ep_sp, ep_orth = 0, 0, 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        y_pred, _ = model(X_b)
        losses = model.compute_loss(y_pred, y_b, lambda_sparse=2e-3, lambda_ortho=1e-2)
        losses['total_loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        ep_loss += losses['total_loss'].item() * len(X_b)
        ep_task += losses['task_loss'].item() * len(X_b)
        ep_sp += losses['sparse_loss'].item() * len(X_b)
        ep_orth += losses['ortho_loss'].item() * len(X_b)
        
    scheduler.step()
    N_tr = len(train_loader.dataset)
    history['loss'].append(ep_loss / N_tr)
    history['task_loss'].append(ep_task / N_tr)
    history['sparse_loss'].append(ep_sp / N_tr)
    history['ortho_loss'].append(ep_orth / N_tr)
    
    if epoch % 20 == 0 or epoch == epochs:
        print(f"Epoch {epoch:02d} | Total Loss: {history['loss'][-1]:.4f} | Task MSE: {history['task_loss'][-1]:.4f}")

## 3. Analysis: Feature Selection & ANOVA Decomposition

In [ ]:
model.eval()
info = model.get_feature_importance()
selected_feats = [i for i, g in enumerate(info['gates']) if g > 0.1]
fs_res = compute_feature_selection_metrics(selected_feats, true_indices, N_FEATURES)

print('=== Feature Selection Performance ===')
print(f'True Features: {true_indices}')
print(f'Selected Features: {selected_feats}')
print(f"Precision: {fs_res['precision']:.4f}")
print(f"Recall:    {fs_res['recall']:.4f}")
print(f"F1-Score:  {fs_res['f1_score']:.4f}")

anova = model.get_anova_variance_decomposition(X_test)
print('\n=== Functional ANOVA Variance Decomposition ===')
print(f"Main Effects:     {anova['main_effect_pct']:.2f}%")
print(f"Pairwise Inter:   {anova['interaction_pct']:.2f}%")
print(f"Deep Residual:    {anova['residual_pct']:.2f}%")

## 4. Visualizations: Loss Curves, Gate Distribution, ANOVA Pie Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(history['task_loss'], label='Task MSE', color='blue')
axes[0].plot(history['sparse_loss'], label='L0 Sparsity', color='green', linestyle='--')
axes[0].plot(history['ortho_loss'], label='DOE Orthogonality', color='red', linestyle=':')
axes[0].set_title('Training Loss Trajectory')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gate values across features
colors = ['red' if i in true_indices else 'gray' for i in range(N_FEATURES)]
axes[1].bar(range(N_FEATURES), info['gates'], color=colors, width=1.0)
axes[1].set_title('Feature Gate Values (Red = True Causal Factors)')
axes[1].set_xlabel('Feature Index (0..499)')
axes[1].set_ylabel('Gate Value [0, 1]')
axes[1].grid(True, alpha=0.3)

# ANOVA Variance Decomposition Pie Chart
labels = ['Main Effects', 'Pairwise Interactions', 'Deep Residual']
sizes = [anova['main_effect_pct'], anova['interaction_pct'], anova['residual_pct']]
axes[2].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=['#3498db', '#2ecc71', '#e74c3c'])
axes[2].set_title('ANOVA Variance Attribution')

plt.tight_layout()
plt.savefig('doenet_analysis_plot.png', dpi=200)
plt.show()
print('Plot saved to doenet_analysis_plot.png')